# Zalo AI Challenge 2026 — LifeMate: Baseline Solution

## 1. Tổng quan Kiến trúc bài toán LifeMate

Cuộc thi **ZAC 2026 - LifeMate** yêu cầu xây dựng một trợ lý ảo thông minh (Voice Agent) có khả năng lắng nghe câu lệnh giọng nói tiếng Việt của người dùng, hiểu ngữ cảnh thời gian/địa điểm/ứng dụng đã cài đặt và thực thi chuỗi **Tool Calls** để cập nhật trạng thái hệ thống (`state_hash`).

### Luồng xử lý End-to-End (Pipeline):
1. **Input**: File âm thanh giọng nói (`.wav`, 16kHz mono) + Bối cảnh `context` (`current_time`, `user_profile`, `installed_apps`) + Lịch sử hội thoại `history`.
2. **ASR Module**: Chuyển đổi giọng nói thành văn bản tiếng Việt (`audio_text`).
3. **Agentic Decision Engine**: Phân tích ý định (Intent), trích xuất tham số thực thể (Entities/Parameters), đối chiếu với danh mục Tool Schema (`tools_catalog.json`) và các app được phép gọi (`installed_apps`).
4. **Tool Execution Runtime (`zaic_tools.ToolEnv`)**: Thực thi tuần tự các tool call trong phiên làm việc cô lập, cập nhật trạng thái ứng dụng trong bộ nhớ.
5. **Submission Format**: Xuất ra file `submission.jsonl` chứa `id`, `tool_calls` và mã băm trạng thái cuối `state_hash`.

In [ ]:
# Setup environment & verify packages
import sys
import os
from pathlib import Path
import json
import torch

# Check device acceleration (Apple Silicon MPS / Nvidia CUDA / CPU)
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"PyTorch: {torch.__version__} | Device: {device}")

# Add zaic_tools to system path
TRAIN_ROOT = Path("train+public_test/train").resolve()
if str(TRAIN_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAIN_ROOT))

from zaic_tools import ToolEnv, schema_validate
from zaic_tools.utils import state_hash
print("zaic_tools imported successfully!")

## 2. Khám phá Dữ liệu & Môi trường ToolEnv

In [ ]:
# Read and inspect first training sample
with open("train+public_test/train/train.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print(f"Sample ID: {sample['id']}")
print(f"Audio path: {sample['audio']}")
print(f"Audio text (Ground Truth): {sample['audio_text']}")
print(f"Context current_time: {sample['context']['current_time']}")
print(f"Installed Apps count: {len(sample['context']['installed_apps'])}")

print("\nGround Truth Tool Calls:")
for c in sample['label']['tool_calls']:
    print(f" -> {c['name']}({c['arguments']})")

In [ ]:
# Initialize ToolEnv session for train_0001
env = ToolEnv("train", sample["id"], root="train+public_test")
init_hash = env.get_state_hash()
print(f"Initial state_hash: {init_hash}")

# Replay ground truth calls
for call in sample["label"]["tool_calls"]:
    res = env.call(call["name"], call["arguments"])
    print(f"Executed {call['name']} -> ok: {res.get('ok')}")

final_hash = env.get_state_hash()
print(f"Final state_hash: {final_hash}")

## 3. Triển khai Baseline Agent

Chúng ta nhập mã nguồn Baseline đã được tối ưu hóa từ file [`baseline.py`](baseline.py).

In [ ]:
from baseline import HeuristicBaselineAgent, evaluate_agent, generate_submission

agent = HeuristicBaselineAgent()
print("HeuristicBaselineAgent initialized!")

## 4. Đánh giá thử nghiệm trên tập Train

In [ ]:
train_path = Path("train+public_test/train/train.jsonl")
results = evaluate_agent(agent, train_path, split="train", max_samples=150)
print(f"\n>>> Train Accuracy on 150 samples: {results['accuracy']*100:.2f}%")

## 5. Sinh File nộp bài Public Test (submission.jsonl) & Kiểm tra tính hợp lệ

In [ ]:
test_path = Path("train+public_test/public_test/public_test.jsonl")
output_file = Path("submission.jsonl")

generate_submission(agent, test_path, output_file, split="public_test")

# Hiển thị 3 dòng đầu tiên của file submission
with open(output_file, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline().strip())